In [1]:
import io
import json
import unicodedata
import xlrd
from openpyxl import load_workbook
from pptx import Presentation
from docx import Document
import json
import pdfplumber
from lxml import etree
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pandas as pd
import faiss
import sys
import win32com.client
from pathlib import Path
from operator import itemgetter
from llama_index.core.schema import Document as llamadoc
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage.index_store import SimpleIndexStore
from llama_index.core.storage.kvstore.simple_kvstore import SimpleKVStore
from bs4 import BeautifulSoup, Tag, NavigableString
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from IPython.display import Markdown, display

c:\Users\txcjs\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\txcjs\anaconda3\envs\myenv\lib\site-packages\opentelemetry\proto\collector\trace\v1\trace_service_pb2_grpc.py:26: RuntimeWarning: The grpc package installed is at version 1.48.2, but the generated code in opentelemetry/proto/collector/trace/v1/trace_service_pb2_grpc.py depends on grpcio>=1.63.2. Please upgrade your grpc module to grpcio>=1.63.2 or downgrade your generated code using grpcio-tools<=1.48.2. This warning will become an error in 1.65.0, scheduled for release on June 25, 2024.
  warnings.warn(


In [2]:
Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2") # MUST BE SAME AS THE ONE USED FOR INDEXING
embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2")
# Set Ollama as the default LLM globally
Settings.llm = Ollama(model="llama3.2:latest", context_window=4096, timeout=120)
def extract_text_from_pdf(file_path):
    """Extracts .pdf files"""
    output = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            blocks = []

            # Extract all tables with bbox
            tables = page.find_tables()
            for table in tables:
                table_bbox = table.bbox
                table_content = []
                for row in table.extract():
                    row_text = " | ".join(cell.strip() if cell else "" for cell in row)
                    table_content.append(f"| {row_text} |")
                blocks.append({
                    'type': 'table',
                    'top': table_bbox[1],
                    'bottom': table_bbox[3],
                    'content': "\n".join(table_content)
                })

            # Extract all words
            words = page.extract_words()
            # Group words into lines by their vertical position (rounded)
            lines_map = {}
            for word in words:
                top = round(word['top'], 1)
                if top not in lines_map:
                    lines_map[top] = []
                lines_map[top].append(word)

            # Convert lines_map to list of text blocks
            for top, word_group in lines_map.items():
                line_text = " ".join(w['text'] for w in sorted(word_group, key=lambda w: w['x0']))
                # Check if line overlaps any table
                in_table = False
                for t in blocks:
                    if t['type'] == 'table' and t['top'] <= top <= t['bottom']:
                        in_table = True
                        break
                if not in_table:
                    blocks.append({
                        'type': 'text',
                        'top': top,
                        'content': line_text
                    })

            # Sort blocks by Y position
            blocks_sorted = sorted(blocks, key=itemgetter('top'))

            page_output = [block['content'] for block in blocks_sorted]
            output.append("\n".join(page_output))

    return "\n\n".join(output)


def extract_text_from_doc(file_path):
    """Extracts .doc files, ignores images"""

    # Have to open the file in background because its old -_-
    word = win32com.client.Dispatch("Word.Application")
    word.Visible = False
    doc = word.Documents.Open(file_path)

    full_text = []
    content = doc.Content
    start = content.Start
    end = content.End
    bullets = {'•', '‣', '·', '‧', '–', '-', '*', '', '●', '■', '♦', '\uf0b7', 'o'}

    while start < end:
        current_range = doc.Range(start, start + 1)
        # Extract tables
        if current_range.Tables.Count > 0:
            table = current_range.Tables(1)
            table_text = []
            for row in table.Rows:
                row_text = []
                for cell in row.Cells:
                    cell_text = cell.Range.Text.strip().replace('\r', '').replace('\x07', '')
                    row_text.append(cell_text)
                table_text.append(' | '.join(row_text))
            full_text.append('\n'.join(table_text))
            start = table.Range.End

        # Extract paragraphs
        elif current_range.Paragraphs.Count > 0:
            para_range = current_range.Paragraphs(1).Range
            para_text = para_range.Text.strip().replace('\r', '').replace('\x07', '')
            para_text = unicodedata.normalize("NFKC", para_text) # Normalize to raw text

            if para_text:
                list_format = para_range.ListFormat
                indent = ''

                '''This chunk of code is to deal with microsoft word lists'''
                if list_format.ListType != 0:
                    # Get list indent level and marker
                    level = max(list_format.ListLevelNumber, 1)
                    indent = '    ' * (level - 1)
                    marker = list_format.ListString.strip()

                    # Some markers are invisible or from Wingdings/Symbol font (like '\uf0b7')
                    # These don't render well, so we substitute a standard bullet
                    if not marker or not marker.isprintable() or ord(marker[0]) >= 0xF000:
                        marker = '•'
                    para_text = f"{indent}{marker} {para_text}"

                else:
                    stripped = para_text.lstrip()
                    # Now check if the first character is a bullet point
                    if stripped and stripped[0] in bullets:
                        para_text = f"• {stripped[1:].lstrip()}"

                full_text.append(para_text)

            start = para_range.End
        else:
            start = current_range.End  # Fallback to avoid infinite loop

     # Extract text from shapes
    for shape in doc.Shapes:
        if shape.TextFrame.HasText:
            shape_text = shape.TextFrame.TextRange.Text.strip().replace('\r', '').replace('\x07', '')
            if shape_text:
                full_text.append("[Shape Text] " + shape_text)

    for ishape in doc.InlineShapes:
        if hasattr(ishape, "TextFrame") and ishape.TextFrame.HasText:
            shape_text = ishape.TextFrame.TextRange.Text.strip().replace('\r', '').replace('\x07', '')
            if shape_text:
                full_text.append("[Inline Shape Text] " + shape_text)
                
    doc.Close(False)
    word.Quit()
    return '\n'.join(full_text)


def extract_text_from_docx(file_path):
    """Extracts .docx files, ignores images"""
    doc = Document(file_path)
    full_text = []

    # Get the raw XML of the Word doc so we can manually look at paragraphs, tables, etc.
    doc_xml = doc.element
    namespaces = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}

    for element in doc_xml.body:
        # Extract text
        if element.tag == etree.QName(namespaces['w'], 'p'):
            # Detect if it's a list item
            num_pr = element.find('.//w:numPr', namespaces)
            is_list = num_pr is not None

            # Extract hyperlink stuff
            hyperlink = element.find('.//w:hyperlink', namespaces)
            if hyperlink is not None:

                # Hyperlink text
                texts = hyperlink.findall('.//w:t', namespaces)
                link_text = ''.join(t.text for t in texts if t.text)

                # Get the hyperlink target
                r_id = hyperlink.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
                if r_id:
                    rels = doc.part.rels
                    url = rels[r_id]._target if r_id in rels else ''
                    plain_text = f"[{link_text}]({url})"
                else:
                    # No url found
                    plain_text = link_text
            else:
                # Plain text
                texts = element.findall('.//w:t', namespaces)
                plain_text = ''.join(t.text for t in texts if t.text)

            plain_text = plain_text.strip()
            if plain_text:
                if is_list:
                    plain_text = f"• {plain_text}"
                full_text.append(plain_text)

        # Extract tables
        elif element.tag == etree.QName(namespaces['w'], 'tbl'):
            for row in element.findall('.//w:tr', namespaces):
                cells = row.findall('.//w:tc', namespaces)
                row_cells = []
                for cell in cells:
                    texts = cell.findall('.//w:t', namespaces)
                    cell_text = ''.join(t.text for t in texts if t.text).strip()
                    row_cells.append(cell_text)
                formatted_row = '| ' + ' | '.join(row_cells) + ' |'
                full_text.append(formatted_row)

    return '\n'.join(full_text)


def extract_text_from_xls(file_path):
    """Extracts .xls file, ignoring formulas"""
    book = xlrd.open_workbook(file_path)
    all_text = ""

    for sheet in book.sheets():
        all_text += f"--- Sheet: {sheet.name} ---\n"
        for row_idx in range(sheet.nrows):
            row_values = sheet.row_values(row_idx)
            line_parts = []
            for val in row_values:
                if isinstance(val, float):
                    line_parts.append(f"{val:.7g}")  # Prevent floating points stuff from going crazy
                else:
                    line_parts.append(str(val).strip())
            all_text += "\t".join(line_parts) + "\n"
        all_text += "\n"
    
    return all_text


def extract_text_from_xlsx(file_path):
    """Extracts .xlsx files, ignoring formulas"""
    wb = load_workbook(filename=file_path, data_only=True) # Don't extract formulas
    all_text = ""

    for sheet in wb.sheetnames:
        ws = wb[sheet]
        all_text += f"--- Sheet: {sheet} ---\n"
        for row in ws.iter_rows():
            row_values = []
            for cell in row:
                value = str(cell.value) if cell.value is not None else ""
                row_values.append(value)
            all_text += "\t".join(row_values).rstrip() + "\n"
        all_text += "\n"
    
    return all_text


def extract_text_from_csv(file_path):
    """Extracts .csv files"""
    df = pd.read_csv(file_path)
    return df.to_string(index=False)


def extract_text_from_txt(file_path):
    """Extracts .txt files"""
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()
    

def extract_text_from_html(file_path):
    """Extract .html files"""
    with open(file_path, 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file, 'html.parser')

    def format_element(element):
        """
        Apply custom formatting to certain HTML elements:
        - <li>: Format as list item with a dash
        - <table>: Format as a tab-separated grid
        - <a>: Replace with "text (href)" format
        """
        if element.name == 'li':
            return f"- {element.get_text(' ', strip=True)}\n"

        elif element.name == 'table':
            table_text = []
            for row in element.find_all('tr'):
                row_text = []
                for cell in row.find_all(['td', 'th']):
                    # Handles cells in tables
                    cell_text = cell.get_text(" ", strip=True).replace('\r', '').replace('\x07', '')
                    row_text.append(cell_text)
                table_text.append(' | '.join(row_text))
            return '\n'.join(table_text) + '\n'
        
        elif element.name == 'a' and element.has_attr('href'):
            return f"{element.get_text(strip=True)} ({element['href']})"
        else:
            return None

    def traverse(node):
        """Traverse the HTML tree and extract formatted text."""
        output = ''

        for child in node.children:
            if isinstance(child, NavigableString):
                output += child
            elif isinstance(child, Tag):
                if child.name in ['script', 'style']:
                    continue
                # Deal with hyperlinks
                if child.name == 'a' and child.has_attr('href'):
                    output += format_element(child)
                elif child.name in ['li', 'table']:
                    output += format_element(child)
                else:
                    output += traverse(child)
        return output

    body = soup.body or soup  # Fallback if the text is not associated with a <body> thing
    formatted_text = traverse(body)
    return "\n".join(line.strip() for line in formatted_text.splitlines() if line.strip())


def extract_text_from_ppt(ppt_path):
    """Extract text and speaker notes from .ppt"""
    powerpoint = win32com.client.Dispatch("PowerPoint.Application")
    powerpoint.Visible = 1

    presentation = powerpoint.Presentations.Open(ppt_path, WithWindow=False)
    all_text = []

    for i, slide in enumerate(presentation.Slides, start=1):
        slide_text = [f"--- Slide {i} ---"]

        for shape in slide.Shapes:
            # Handle text and bullet lists
            if shape.HasTextFrame:
                tf = shape.TextFrame
                if tf.HasText:
                    paragraphs = []
                    for paragraph in tf.TextRange.Paragraphs():
                        text = paragraph.Text.strip().replace('\r', '')
                        if text:
                            bullet = "- " if paragraph.ParagraphFormat.Bullet.Type != 0 else ""
                            paragraphs.append(bullet + text)
                    if paragraphs:
                        slide_text.append("\n".join(paragraphs))

            # Handle tables
            if shape.HasTable:
                table = shape.Table
                table_text = []
                for row in range(1, table.Rows.Count + 1):
                    row_text = []
                    for col in range(1, table.Columns.Count + 1):
                        cell = table.Cell(row, col)
                        cell_text = cell.Shape.TextFrame.TextRange.Text.strip().replace('\r', '').replace('\x07', '')
                        row_text.append(cell_text)
                    table_text.append(" | ".join(row_text))
                slide_text.append("\n".join(table_text))

        # Speaker Notes
        if slide.NotesPage.Shapes.Placeholders.Count >= 2:
            notes_shape = slide.NotesPage.Shapes.Placeholders(2)
            if notes_shape.HasTextFrame and notes_shape.TextFrame.HasText:
                notes = notes_shape.TextFrame.TextRange.Text.strip().replace('\r', '')
                if notes:
                    slide_text.append(f"[Notes] {notes}")

        all_text.append("\n".join(slide_text))

    presentation.Close()
    powerpoint.Quit()

    return "\n\n".join(all_text)


def is_bullet_paragraph(paragraph):
    """
    Check if a paragraph has bullet formatting by inspecting XML.
    We have to do this since python-pptx cannot detect bullets natively :(
    """
    pPr = paragraph._element.pPr
    return pPr is not None and pPr.find(".//a:buChar", namespaces={'a': 'http://schemas.openxmlformats.org/drawingml/2006/main'}) is not None

def extract_text_from_pptx(file_path):
    """Extract text and notes from a .pptx file."""
    presentation = Presentation(file_path)
    all_text = []

    for i, slide in enumerate(presentation.slides, start=1):
        slide_text = [f"--- Slide {i} ---"]

        for shape in slide.shapes:
            # Handle text and bullet lists
            if shape.has_text_frame:
                paragraphs = []
                for para in shape.text_frame.paragraphs:
                    text = para.text.strip()
                    if not text:
                        continue

                    if is_bullet_paragraph(para):
                        indent = "  " * para.level
                        paragraphs.append(f"{indent}- {text}")
                    else:
                        paragraphs.append(text)

                if paragraphs:
                    slide_text.append("\n".join(paragraphs))

            # Handle Tables
            if shape.shape_type == 19:  # MSO_SHAPE_TYPE.TABLE
                table = shape.table
                table_text = []
                for row in table.rows:
                    row_text = []
                    for cell in row.cells:
                        cell_text = cell.text.strip().replace('\r', '').replace('\x07', '')
                        row_text.append(cell_text)
                    table_text.append(' | '.join(row_text))
                slide_text.append('\n'.join(table_text))

        # Speaker Notes
        notes_slide = slide.notes_slide if slide.has_notes_slide else None
        if notes_slide:
            notes_text = notes_slide.notes_text_frame.text.strip()
            if notes_text:
                slide_text.append(f"[Notes] {notes_text}")

        all_text.append("\n\n".join(slide_text))

    return "\n\n".join(all_text)

def extract_text_from_md(file_path):
    """Extract .md file"""
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

ALL_DEPARTMENTS = [
    "Human Resource", "Admin & Operations", "Project Management", "Procurement",
    "IT", "Marketing", "Business Development", "Finance", "Service Delivery"
]
ALL_COUNTRIES = [
    "Singapore", "United Kingdom", "United States", "Thailand", "Indonesia",
    "Korea", "China", "Japan", "Vietnam", "Myanmar"
]

# 2. Chunking

def split_into_documents(text, chunk_size=1000, chunk_overlap=200, title="Untitled", source="unknown.txt", departments=None, countries=None):
    """
    Splits text into chunks and returns them as LlamaIndex Document objects with metadata.

    Parameters:
        text (str): The full input text to split.
        chunk_size (int): Max characters per chunk.
        chunk_overlap (int): Characters to overlap between chunks.
        title (str): Title of the source document.
        source (str): Filename of the source document.
        departments (list): List of departments that can see the document.
        countries (list): List of countries that can see the document.

    Returns:
        List[Document]: Chunked Document objects with metadata.
    """
    if departments is None:
            departments = []
    if countries is None:
        countries = []

    splitter = RecursiveCharacterTextSplitter( # Used RecussiveCharacterTextSplitter because it's good at identifying paragraphs and natural sections
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_text(text)
    print(f"Total chunks: {len(chunks)}\n")

    documents = []
    for i, chunk in enumerate(chunks):
        metadata = {
            "chunk": i,
            "title": title,
            "source": source,
        }
        for dept in ALL_DEPARTMENTS:
            metadata[dept] = str(dept in departments)
        for country in ALL_COUNTRIES:
            metadata[country] = str(country in countries)

        doc = llamadoc(text=chunk, metadata=metadata)
        documents.append(doc)

    return documents
def extract_text_from_file(file_path):
    """Extract text based on file extension"""
    file_extension = file_path.lower().split('.')[-1]
    
    if file_extension == "docx":
        return extract_text_from_docx(file_path)
    elif file_extension == "doc":
        return extract_text_from_doc(file_path)
    elif file_extension == "pdf":
        return extract_text_from_pdf(file_path)
    elif file_extension == "pptx":
        return extract_text_from_pptx(file_path)
    elif file_extension == "ppt":
        return extract_text_from_ppt(file_path)
    elif file_extension == "xls":
        return extract_text_from_xls(file_path)
    elif file_extension == "xlsx":
        return extract_text_from_xlsx(file_path)
    elif file_extension == "csv":
        return extract_text_from_csv(file_path)
    elif file_extension == "txt":
        return extract_text_from_txt(file_path)
    elif file_extension == "html":
        return extract_text_from_html(file_path)
    elif file_extension == "md":
        return extract_text_from_md(file_path)
    else:
        raise ValueError(f"Unsupported file type: {file_extension}")

        

In [3]:
file = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Presentation\pipeline\data\raw_data\3_Offboarding Process on Clean Desk Policy_150125.pdf"
extracted_text = extract_text_from_file(str(file))
print(extracted_text)


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


15th
Date: JAN 2025
Offboarding Clean Desk and Digital Handover Policy
This policy outlines the steps employees must take to maintain a clean and organized workspace and ensure
all necessary digital files are properly backed up and handed over before their final day of work.
Clean Desk Policy
Employees must ensure their workspace is clear of personal and unnecessary items by the end of their final
working day. This includes:
1. Removal of Personal Belongings:
Take home all personal items such as photos, decorations, and personal stationery etc.
o
Check and empty all drawers, cabinets, and other storage areas for personal belongings.
o
Bring back/ throw away any food or drinks you stored in the office refrigerator.
o
2. Organizing Work Materials:
Sort through physical documents. Shred or dispose of sensitive documents no longer
o
needed. Remove all name cards and old files outside the bin near the lift area.
Return all company property (e.g., laptop, keyboard, employment card, medical c

In [4]:
documents = split_into_documents(extracted_text, title=file, source=file, departments=["Human Resource", ""], countries=["Singapore"])

Total chunks: 5



In [5]:
PROJECT_ROOT = Path.cwd().parent.parent
print(f"Project root: {PROJECT_ROOT}")
PERSIST_DIR = PROJECT_ROOT / 'pipeline' / 'data' / 'ChromaDB'
print(f"Persist directory: {PERSIST_DIR}")

Project root: c:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Presentation
Persist directory: c:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\Presentation\pipeline\data\ChromaDB


In [9]:
from sentence_transformers import SentenceTransformer
from llama_index.core.base.embeddings.base import BaseEmbedding
import numpy as np
from typing import List
from typing import Union
# Load model locally
model = SentenceTransformer("intfloat/e5-large-v2")

# ChromaDB embedding function
class HFChromaEmbedding:
    def __init__(self, model):
        self.model = model

    def __call__(self, input: Union[str, List[str]]) -> Union[List[float], List[List[float]]]:
        if isinstance(input, str):
            input = [f"passage: {input}"]
            return self.model.encode(input, convert_to_numpy=True).tolist()[0]
        else:
            input = [f"passage: {text}" for text in input]
            return self.model.encode(input, convert_to_numpy=True).tolist()

    def name(self) -> str:
        return "HFChromaEmbedding-e5-large-v2"

chroma_embedding_fn = HFChromaEmbedding(model)

# LlamaIndex embedding class
class HFLlamaEmbedding(BaseEmbedding):
    model: SentenceTransformer
    def __init__(self, model):
        super().__init__(model=model)

    def _get_text_embedding(self, text: str) -> List[float]:
        return self.model.encode(f"passage: {text}", convert_to_numpy=True).tolist()

    def _get_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode([f"passage: {t}" for t in texts], convert_to_numpy=True).tolist()

    def _get_query_embedding(self, query: str) -> List[float]:
        return self.model.encode(f"query: {query}", convert_to_numpy=True).tolist()

    async def _aget_query_embedding(self, query: str) -> List[float]:
        return self._get_query_embedding(query)

llama_embedding_fn = HFLlamaEmbedding(model)

In [10]:
def build_or_append_index(documents,embed_model,persist_dir="pipeline/data/EmbeddedChroma",collection_name="quickstart"):
    """
    Create or append to a ChromaDB + LlamaIndex index.

    Parameters:
        documents (List[Document]): New documents to insert
        embed_model (BaseEmbedding): Embedding model (e.g., HuggingFaceEmbedding)
        persist_dir (str or Path): Directory where ChromaDB data is stored
        collection_name (str): Name of the ChromaDB collection
    """
    persist_dir = Path(persist_dir)
    persist_dir.mkdir(parents=True, exist_ok=True)

    # Initialize ChromaDB persistent client
    db = chromadb.PersistentClient(path=str(persist_dir))
    chroma_collection = db.get_or_create_collection(collection_name, embedding_function=chroma_embedding_fn)
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    

    # Check if collection already has data
    existing_count = chroma_collection.count()
    if existing_count > 0:
        print("Loading existing ChromaDB index...")
        index = VectorStoreIndex.from_vector_store(
            vector_store, storage_context=storage_context, embed_model=embed_model
        )
        print("Total docs in index before append:", existing_count)
        # Append new documents
        parser = SimpleNodeParser()
        nodes = parser.get_nodes_from_documents(documents)
        index.insert_nodes(nodes)
    else:
        print("Creating new ChromaDB index...")
        index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context, embed_model=embed_model
        )

    # ChromaDB persists automatically, but you can force a flush if needed
    print("Saving new data...")
    print("Total docs in index:", chroma_collection.count())

    return index

In [19]:
build_or_append_index(doc2, llama_embedding_fn, persist_dir=PERSIST_DIR, collection_name="test")

Loading existing ChromaDB index...
Total docs in index before append: 16
Saving new data...
Total docs in index: 20


In [14]:
file = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\raw_data - Copy\Follow Ups_Importance.pdf"
extracted_text2 = extract_text_from_file(str(file))
doc2 = split_into_documents(extracted_text2, title=file, source=file, departments=["Human Resource"], countries=["Singapore"])

Total chunks: 4



In [16]:
file = r"C:\Users\txcjs\OneDrive\Documents\Homework\Yr 3.1\ICP\raw_data - Copy\leave policy.docx"
extracted_text1 = extract_text_from_file(str(file))
doc1 = split_into_documents(extracted_text1, title=file, source=file, departments=["Human Resource"], countries=["Singapore"])

Total chunks: 11



In [ ]:
db = chromadb.PersistentClient(path=str(PERSIST_DIR))
chroma_collection = db.get_or_create_collection("quicksatrt", embedding_function=chroma_embedding_fn)

In [9]:
# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context, embed_model=llama_embedding_fn
        )


In [11]:
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# load your index from stored vectors
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context, embed_model=llama_embedding_fn
)

In [18]:
build_or_append_index(documents, embed_model=llama_embedding_fn, persist_dir=PERSIST_DIR, collection_name="sdaffasdfa")

Creating new ChromaDB index...
Saving new data...
Total docs in index: 5


In [9]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client
db = chromadb.PersistentClient(path=str(PERSIST_DIR))

# get collection
chroma_collection = db.get_or_create_collection("hello")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# load your index from stored vectors
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context, embed_model=llama_embedding_fn
)

In [10]:
# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context, embed_model=llama_embedding_fn
        )


In [37]:
# Chroma
db = chromadb.PersistentClient(path=str(PERSIST_DIR))
collection = db.get_or_create_collection(
    name="hello",
    embedding_function=chroma_embedding_fn
)

# LlamaIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store,
    storage_context=storage_context,
    embed_model=llama_embedding_fn
)


ValueError: Expected EmbeddingFunction.__call__ to have the following signature: odict_keys(['self', 'input']), got odict_keys(['self', 'texts'])
Please see https://docs.trychroma.com/guides/embeddings for details of the EmbeddingFunction interface.
Please note the recent change to the EmbeddingFunction interface: https://docs.trychroma.com/deployment/migration#migration-to-0.4.16---november-7,-2023 


In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client
db = chromadb.PersistentClient(path=str(PERSIST_DIR))

# get collection
chroma_collection = db.get_or_create_collection("hello")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# load your index from stored vectors
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context, embed_model=embed_model
)

In [36]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client
db = chromadb.PersistentClient(path=str(PERSIST_DIR))

# get collection
chroma_collection = db.get_or_create_collection("hello")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context, embed_model=model
        )


AssertionError: 

In [25]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client
db = chromadb.PersistentClient(path=str(PERSIST_DIR))

# get collection
chroma_collection = db.get_or_create_collection("hello")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# load your index from stored vectors
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context, embed_model=embed_model
)


In [3]:

def build_or_append_index(documents,embed_model,persist_dir="pipeline/data/EmbeddedChroma",collection_name="quickstart"):
    """
    Create or append to a ChromaDB + LlamaIndex index.

    Parameters:
        documents (List[Document]): New documents to insert
        embed_model (BaseEmbedding): Embedding model (e.g., HuggingFaceEmbedding)
        persist_dir (str or Path): Directory where ChromaDB data is stored
        collection_name (str): Name of the ChromaDB collection
    """
    persist_dir = Path(persist_dir)
    persist_dir.mkdir(parents=True, exist_ok=True)

    # Initialize ChromaDB persistent client
    db = chromadb.PersistentClient(path=str(persist_dir))
    chroma_collection = db.get_or_create_collection(collection_name, embed_model=embed_model)
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    

    # Check if collection already has data
    existing_count = chroma_collection.count()
    if existing_count > 0:
        print("Loading existing ChromaDB index...")
        index = VectorStoreIndex.from_vector_store(
            vector_store, storage_context=storage_context, embed_model=embed_model
        )
        print("Total docs in index before append:", existing_count)
        # Append new documents
        parser = SimpleNodeParser()
        nodes = parser.get_nodes_from_documents(documents)
        index.insert_nodes(nodes)
    else:
        print("Creating new ChromaDB index...")
        index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context, embed_model=embed_model
        )

    # ChromaDB persists automatically, but you can force a flush if needed
    print("Saving new data...")
    print("Total docs in index:", chroma_collection.count())

    return index

In [4]:
build_or_append_index(documents, embed_model, persist_dir=PERSIST_DIR, collection_name="quickstart")

NameError: name 'documents' is not defined

In [14]:
chroma_collection.count()

5

In [42]:
filtered = chroma_collection.get(where={"title": "[Singapore] toilet_break_policy.docx"})

for i, doc_id in enumerate(filtered["ids"]):
    print(f"ID: {doc_id}")
    print(f"Metadata: {filtered['metadatas'][i]}")
    print("-" * 40)

ID: 8052d00d-fb3a-46d8-b1df-4e95fa17961a
Metadata: {'Indonesia': 'False', 'Singapore': 'True', 'United States': 'False', 'Service Delivery': 'False', 'Business Development': 'False', 'United Kingdom': 'False', 'Japan': 'False', 'chunk': 0, 'doc_id': '15aba6d8-304d-4abf-a870-60b23cbd6480', 'document_id': '15aba6d8-304d-4abf-a870-60b23cbd6480', 'Procurement': 'False', 'Korea': 'False', 'Human Resource': 'False', 'title': '[Singapore] toilet_break_policy.docx', 'Marketing': 'False', '_node_type': 'TextNode', 'ref_doc_id': '15aba6d8-304d-4abf-a870-60b23cbd6480', 'Myanmar': 'False', 'source': '[Singapore] toilet_break_policy.docx', 'IT': 'False', 'Admin & Operations': 'False', 'China': 'False', 'Finance': 'False', '_node_content': '{"id_": "8052d00d-fb3a-46d8-b1df-4e95fa17961a", "embedding": null, "metadata": {"chunk": 0, "title": "[Singapore] toilet_break_policy.docx", "source": "[Singapore] toilet_break_policy.docx", "Human Resource": "False", "Admin & Operations": "False", "Project Manag

In [21]:
doc_to_update = chroma_collection.get(limit=500)
doc_to_update

{'ids': ['eb4f84cf-3e0f-47de-a1eb-5a6255a1846d',
  'c0a6de77-6910-4661-b5fe-fef9d28a9c3a',
  '750cd508-3e95-44a8-a754-27acee50c2ee',
  '13456351-fa1e-4560-8f6b-574b1526e4d0',
  '6c6cd548-16f0-496f-bfaf-2ea45c7e9e8a',
  '2cd7b507-5348-4e55-bdc7-8084ae74f8c2',
  'dc8a2772-804c-4d0a-a585-024dfe59b830',
  '5a468707-ee09-4e44-bb23-4ccf77b70ae1',
  '04962d4f-1b67-40b1-9314-74bbf11f0576',
  '5465b1b4-5562-40fe-bdea-3ebd70ad71cd'],
 'embeddings': None,
 'documents': ['15th\nDate: JAN 2025\nOffboarding Clean Desk and Digital Handover Policy\nThis policy outlines the steps employees must take to maintain a clean and organized workspace and ensure\nall necessary digital files are properly backed up and handed over before their final day of work.\nClean Desk Policy\nEmployees must ensure their workspace is clear of personal and unnecessary items by the end of their final\nworking day. This includes:\n1. Removal of Personal Belongings:\nTake home all personal items such as photos, decorations, and 

In [26]:
print(chroma_collection._embedding_function)


In [22]:
Settings.embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-large-v2")

existing = chroma_collection.get(ids=["eb4f84cf-3e0f-47de-a1eb-5a6255a1846d"])
original_doc = existing["documents"][0]
original_metadata = existing["metadatas"][0]

updated_metadata = original_metadata.copy()
updated_metadata["Myanmar"] = "True"
updated_metadata["Vietnam"] = "True"

chroma_collection.update(
    ids=["eb4f84cf-3e0f-47de-a1eb-5a6255a1846d"],
    documents=[original_doc],
    metadatas=[updated_metadata]
)

In [23]:
existing = chroma_collection.get(ids=["eb4f84cf-3e0f-47de-a1eb-5a6255a1846d"])
original_doc = existing["documents"][0]
original_metadata = existing["metadatas"][0]

In [24]:
original_metadata

{'Thailand': 'False',
 'Japan': 'False',
 'Finance': 'False',
 'Myanmar': 'True',
 'Human Resource': 'True',
 'Marketing': 'False',
 '_node_content': '{"id_": "eb4f84cf-3e0f-47de-a1eb-5a6255a1846d", "embedding": null, "metadata": {"chunk": 0, "title": "C:\\\\Users\\\\txcjs\\\\OneDrive\\\\Documents\\\\Homework\\\\Yr 3.1\\\\ICP\\\\Presentation\\\\pipeline\\\\data\\\\raw_data\\\\3_Offboarding Process on Clean Desk Policy_150125.pdf", "source": "C:\\\\Users\\\\txcjs\\\\OneDrive\\\\Documents\\\\Homework\\\\Yr 3.1\\\\ICP\\\\Presentation\\\\pipeline\\\\data\\\\raw_data\\\\3_Offboarding Process on Clean Desk Policy_150125.pdf", "Human Resource": "True", "Admin & Operations": "False", "Project Management": "False", "Procurement": "False", "IT": "False", "Marketing": "False", "Business Development": "False", "Finance": "False", "Service Delivery": "False", "Singapore": "True", "United Kingdom": "False", "United States": "False", "Thailand": "False", "Indonesia": "False", "Korea": "False", "China

In [13]:
updated_metadata

{'Procurement': 'True',
 'Marketing': 'True',
 'Thailand': 'False',
 '_node_type': 'TextNode',
 'Project Management': 'True',
 'Singapore': 'True',
 'United States': 'False',
 'Japan': 'False',
 'source': '[Singapore] toilet_break_policy.docx',
 'Finance': 'False',
 'Indonesia': 'False',
 'ref_doc_id': 'ddc195e3-74fc-4aa7-98fd-c8cc1a07a4fb',
 'Service Delivery': 'True',
 'United Kingdom': 'False',
 'IT': 'True',
 'chunk': 1,
 'title': 'Updated Title',
 'doc_id': 'ddc195e3-74fc-4aa7-98fd-c8cc1a07a4fb',
 'document_id': 'ddc195e3-74fc-4aa7-98fd-c8cc1a07a4fb',
 'Admin & Operations': 'True',
 'Human Resource': 'True',
 'China': 'False',
 'Korea': 'False',
 '_node_content': '{"id_": "783f1cf9-c016-4069-a6f0-f34cbe29d94a", "embedding": null, "metadata": {"chunk": 1, "title": "[Singapore] toilet_break_policy.docx", "source": "[Singapore] toilet_break_policy.docx", "Human Resource": "True", "Admin & Operations": "True", "Project Management": "True", "Procurement": "True", "IT": "True", "Marketi

In [ ]:
# Step 3: Re-upsert with full metadata
chroma_collection.upsert(
    ids=["my_id"],
    documents=[original_doc],
    metadatas=[updated_metadata]
)

In [10]:
type(filtered)

dict

In [12]:
doc_to_update = chroma_collection.get(limit=1)
doc_to_update["metadatas"][0]

IndexError: list index out of range

In [ ]:
doc_to_update = chroma_collection.get(limit=1)
doc_to_update["metadatas"][0] = {
    **doc_to_update["metadatas"][0],
    **{"author": "Paul Graham"},
}
chroma_collection.update(
    ids=[doc_to_update["ids"][0]], metadatas=[doc_to_update["metadatas"][0]]
)
updated_doc = chroma_collection.get(limit=1)
print(updated_doc["metadatas"][0])

# delete the last document
print("count before", chroma_collection.count())
chroma_collection.delete(ids=[doc_to_update["ids"][0]])
print("count after", chroma_collection.count())

In [16]:
from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
)
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

filters = MetadataFilters(
    filters=[
        MetadataFilter(key="Vietnam", value="True")
    ]
)


# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5)  # MetadataFilters object)

# configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize")

In [17]:
llm2 = Ollama(model="llama3.2:1b", request_timeout=120.0, temperature=0, context_window=4096)

In [18]:
from typing import Generator
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer
)

# query
response = query_engine.query(
    "What is the offboarding process?",
)


In [20]:
import pprint as pp
pp.pprint(response.response)

('The offboarding process involves several steps, including:\n'
 '\n'
 '1. Deleting personal files from company devices to ensure a clean '
 'workspace.\n'
 '\n'
 '2. Archiving important emails and sharing access with relevant team '
 'members.\n'
 '\n'
 '3. Logging out of all personal accounts from company systems.\n'
 '\n'
 '4. Scheduling a review with your supervisor or HR to confirm the handover of '
 'all files and credentials, verify the return of company property, and '
 'conduct a final check of the workspace.\n'
 '\n'
 '5. Transferring all work-related files to designated shared drives or cloud '
 'storage, organizing them logically for easy access.\n'
 '\n'
 '6. Sharing login credentials for any work-related systems, tools, or '
 'platforms with your supervisor or IT department, if applicable.\n'
 '\n'
 '7. Finally, receiving an acknowledgment from HR that you have completed the '
 'offboarding process and a notice about the potential withholding of final '
 'payments until a

In [ ]:
from typing import Generator
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer
)

# query
streaming_response = query_engine.query(
    "What is the offboarding process?",
)

def stream_generator() -> Generator[str, None, None]:
    for token in streaming_response.response_gen:
            yield token



In [ ]:
from llama_index.core import PromptTemplate
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.chat_engine import CondenseQuestionChatEngine

custom_prompt = PromptTemplate(
    """\
Given a conversation (between Human and Assistant) and a follow up message from Human, \
rewrite the message to be a standalone question that captures all relevant context \
from the conversation.

<Chat History>
{chat_history}

<Follow Up Message>
{question}

<Standalone question>
"""
)

# list of `ChatMessage` objects
custom_chat_history = [
    ChatMessage(
        role=MessageRole.USER,
        content="Hello assistant, we are having a insightful discussion about Paul Graham today.",
    ),
    ChatMessage(role=MessageRole.ASSISTANT, content="Okay, sounds good."),
]

query_engine = index.as_query_engine()
chat_engine = CondenseQuestionChatEngine.from_defaults(
    query_engine=query_engine,
    condense_question_prompt=custom_prompt,
    chat_history=custom_chat_history,
    verbose=True,
)

In [30]:
query_engine = index.as_query_engine(response_synthesizer=response_synthesizer)
response = query_engine.query("How long can I go to the toilet for?")
print(response)

A reasonable duration for a toilet break is generally considered to be under 10 minutes.


In [20]:
doc_to_update = chroma_collection.get(limit=500)
doc_to_update

{'ids': ['af3067a7-1f4b-4b56-8a97-2cc196de607a',
  'e826de7a-e2b5-4e57-9cb5-4d527dd95f2b',
  'dcde6823-6899-464e-83ec-87ea1f18a3d6',
  '0e27f77c-f056-4b5a-bba2-2d4ed1e3f3b6',
  '733736f4-68de-4f8f-a641-5fc3533b6fb8'],
 'embeddings': None,
 'documents': ['15th\nDate: JAN 2025\nOffboarding Clean Desk and Digital Handover Policy\nThis policy outlines the steps employees must take to maintain a clean and organized workspace and ensure\nall necessary digital files are properly backed up and handed over before their final day of work.\nClean Desk Policy\nEmployees must ensure their workspace is clear of personal and unnecessary items by the end of their final\nworking day. This includes:\n1. Removal of Personal Belongings:\nTake home all personal items such as photos, decorations, and personal stationery etc.\no\nCheck and empty all drawers, cabinets, and other storage areas for personal belongings.\no\nBring back/ throw away any food or drinks you stored in the office refrigerator.\no\n2. O

In [9]:
filtered = chroma_collection.get(where={"chunk": 334})

for i, doc_id in enumerate(filtered["ids"]):
    print(f"ID: {doc_id}")
    print(f"Text: {filtered['documents'][i][:200]}")
    print(f"Metadata: {filtered['metadatas'][i]}")
    print("-" * 40)

In [8]:
from pathlib import Path
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core import StorageContext

def build_or_append_index(
    documents,
    embed_model,
    persist_dir="pipeline/data/EmbeddedChroma",
    collection_name="quickstart"
):
    """
    Create or append to a ChromaDB + LlamaIndex index.

    Parameters:
        documents (List[Document]): New documents to insert
        embed_model (BaseEmbedding): Embedding model (e.g., HuggingFaceEmbedding)
        persist_dir (str or Path): Directory where ChromaDB data is stored\
        collection_name (str): Name of the ChromaDB collection
    """
    persist_dir = Path(persist_dir)
    persist_dir.mkdir(parents=True, exist_ok=True)

    # Initialize ChromaDB persistent client
    db = chromadb.PersistentClient(path=str(persist_dir))
    chroma_collection = db.get_or_create_collection(collection_name)
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    

    # Check if collection already has data
    existing_count = chroma_collection.count()
    if existing_count > 0:
        print("Loading existing ChromaDB index...")
        index = VectorStoreIndex.from_vector_store(
            vector_store, storage_context=storage_context, embed_model=embed_model
        )
        print("Total docs in index before append:", existing_count)
        # Append new documents
        parser = SimpleNodeParser()
        nodes = parser.get_nodes_from_documents(documents)
        index.insert_nodes(nodes)
    else:
        print("Creating new ChromaDB index...")
        index = VectorStoreIndex.from_documents(
            documents, storage_context=storage_context, embed_model=embed_model
        )

    # ChromaDB persists automatically, but you can force a flush if needed
    print("Saving new data...")
    print("Total docs in index:", chroma_collection.count())

    return index

In [9]:
build_or_append_index(
    documents,
    embed_model,
    persist_dir=str(PERSIST_DIR),
    collection_name="quickstart"
)

Creating new ChromaDB index...
Saving new data...
Total docs in index: 5


In [ ]:
# ...existing code...
for filename in raw_folder.iterdir():
    if not filename.is_file():
        continue
    if filename.name in processed_files:
        continue

    print(f"\nProcessing file: {filename.name}")
    extracted_text = extract_text_from_file(str(filename))

    # --- Read metadata sidecar if exists ---
    meta_path = filename.with_suffix(filename.suffix + ".meta.json")
    departments = []
    countries = []
    if meta_path.exists():
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
            departments = meta.get("departments", [])
            countries = meta.get("countries", [])

    documents = split_into_documents(
        extracted_text,
        title=filename.name,
        source=filename.name,
        departments=departments,
        countries=countries
    )
    build_or_append_index(documents, embed_model, persist_dir=PERSIST_DIR, faiss_path=faiss_file_path.name, embedding_dim=1024)
    new_files.append(filename.name)